# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder)without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/14 15:52:43 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/14 15:52:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3c64ca1f-6487-4206-9027-c2fce5481e2c;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Queries

## Task 1.1

In [6]:
result = spark.sql("""
    SELECT
        pu_zone,
        month(pu_datetime) AS month, 
        COUNT(*) AS row_count
    FROM default.integrated_taxi_trips
    GROUP BY pu_zone, month
""")
result.show()

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       36|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32983|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142707|
|       Rockaway Park|    1|      117|
|           Stapleton|    1|        4|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      341|
|Bay Terrace/Fort ...|    1|       57|
|       Fordham South|    1|      178|
|             Bayside|    1|      111|
|    Garment District|    1|    48093|
|    Bensonhurst East|    1|      145|
|     Cambria Heights|    1|      200|
|Governor's Island...|    1|        1|
|Upper West Side N...|    1|    64234|
|   Kew Gardens Hills|    1|      216|
|Springfield Garde...|    1|      608|
+--------------------+-----+---------+
only showing top 20 rows


## Task 1.2

In [12]:
result = spark.sql(
"""
SELECT
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
        ELSE 'zero_or_null'
  	END AS column_group,
    COUNT(*) AS cnt,
    AVG(trip_distance) AS avg_trip_distance
FROM integrated_taxi_trips
GROUP BY
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
            ELSE 'zero_or_null'
    END;
"""
)
result.show()


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 444243|3.8266784559803266|
|  zero_or_null|2629381|3.9581809210861802|
+--------------+-------+------------------+



In [13]:
# select max and min of prcp
result = spark.sql("""
SELECT
    MAX(prcp) AS max_prcp,
    MIN(prcp) AS min_prcp
FROM integrated_taxi_trips
""")
result.show()

+--------+--------+
|max_prcp|min_prcp|
+--------+--------+
|     9.9|     0.0|
+--------+--------+



In [14]:
result = spark.sql(
"""
SELECT
    FLOOR(prcp) AS prcp_interval,
    COUNT(*) AS cnt,
    AVG(trip_distance) AS avg_trip_distance
FROM integrated_taxi_trips
GROUP BY FLOOR(prcp)
ORDER BY prcp_interval;
"""
)
result.show()

+-------------+-------+------------------+
|prcp_interval|    cnt| avg_trip_distance|
+-------------+-------+------------------+
|         NULL| 341557|  4.26503047466623|
|            0|2604760|3.9137251615924686|
|            1|  70213|3.4771939688602465|
|            2|  24611| 3.564521146582114|
|            3|  23035|3.5641423907898333|
|            4|    410|  5.96612194082359|
|            5|   1861| 4.769494886429503|
|            6|   4003| 3.723162629594363|
|            8|    325| 7.249846145659685|
|            9|   2849| 4.886707621077046|
+-------------+-------+------------------+

